# Ablation A8 & A9 — Lookback Window

**Mục đích**: Đánh giá ảnh hưởng của độ dài lookback window lên hiệu suất dự báo.

Proposed model dùng `lookback=30`. Ablation này thử:
- **A8**: `lookback=7` — ngắn hạn, chỉ nhìn lại 1 tuần
- **A9**: `lookback=60` — dài hạn, nhìn lại 2 tháng

| Variant | Lookback | Ghi chú |
|---------|----------|---------|
| **A8** | 7 ngày | Capture weekly pattern |
| Proposed | 30 ngày | Baseline |
| **A9** | 60 ngày | Capture monthly + seasonal pattern |

Kiến trúc LSTM + Entity Embedding và NUM_COLS giữ nguyên hoàn toàn như Proposed.  
**Output**: `result/ablation_A8_lookback7_summary.csv`, `result/ablation_A9_lookback60_summary.csv`

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings, os
import kagglehub

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

DATA_DIR  = kagglehub.dataset_download("atomicd/retail-store-inventory-and-demand-forecasting")
DATA_PATH = os.path.join(DATA_DIR, "sales_data.csv")
RESULT_DIR = "/kaggle/working/"

TARGET    = 'Units Sold'
TRAIN_END = '2023-06-30'
VAL_END   = '2023-10-31'
HORIZONS  = [7, 14, 28]
LAG       = 7

# ── Lookback variants ────────────────────────────────────────────────────────
LOOKBACK_PROPOSED = 30   # baseline
LOOKBACK_A8       = 7    # A8: short
LOOKBACK_A9       = 60   # A9: long

print(f'Lookbacks: A8={LOOKBACK_A8} | Proposed={LOOKBACK_PROPOSED} | A9={LOOKBACK_A9}')

## 1. Load & Feature Engineering (giống Proposed)

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# ── Static categoricals → Entity Embedding ───────────────────────────────────
CAT_COLS = ['Store ID', 'Product ID', 'Category', 'Region']
cat_vocabs = {}
for col in CAT_COLS:
    uniq = sorted(df_raw[col].unique())
    cat_vocabs[col] = {v: i for i, v in enumerate(uniq)}
    df_raw[col + '_enc'] = df_raw[col].map(cat_vocabs[col])

# ── Time-varying categoricals ─────────────────────────────────────────────────
WEATHER_MAP = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3, 'Windy': 4, 'Stormy': 5}
SEASON_MAP  = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
df_raw['weather_enc'] = df_raw['Weather Condition'].map(WEATHER_MAP).fillna(0).astype(int)
df_raw['season_enc']  = df_raw['Seasonality'].map(SEASON_MAP).fillna(0).astype(int)

# ── Lag & rolling features ────────────────────────────────────────────────────
grp = df_raw.groupby(['Store ID', 'Product ID'])[TARGET]
df_raw['lag_7']           = grp.shift(7)
df_raw['lag_14']          = grp.shift(14)
df_raw['lag_28']          = grp.shift(28)
df_raw['rolling_mean_7']  = grp.transform(lambda x: x.shift(1).rolling(7).mean())
df_raw['rolling_mean_14'] = grp.transform(lambda x: x.shift(1).rolling(14).mean())
df_raw['day_of_week']     = df_raw['Date'].dt.dayofweek
df_raw['day_of_month']    = df_raw['Date'].dt.day
df_raw['month']           = df_raw['Date'].dt.month
df_raw['is_weekend']      = (df_raw['day_of_week'] >= 5).astype(int)
df_raw = df_raw.bfill().fillna(0)

# ── NUM_COLS: giống Proposed hoàn toàn ───────────────────────────────────────
NUM_COLS = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    'Promotion', 'Epidemic',
    'weather_enc', 'season_enc',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
]
TARGET_IDX      = NUM_COLS.index(TARGET)
ENC_COLS        = [c + '_enc' for c in CAT_COLS]
cat_vocab_sizes = {col: len(cat_vocabs[col]) for col in CAT_COLS}
series_keys     = sorted(df_raw.groupby(['Store ID', 'Product ID']).groups.keys())

print(f'Series: {len(series_keys)} | Num features: {len(NUM_COLS)}')

## 2. Model — LSTM + Entity Embedding (giống Proposed)

In [ ]:
def build_model(lookback, n_num, cat_vocab_sizes, horizon, lstm_units=64, dropout=0.2):
    num_input = layers.Input(shape=(lookback, n_num), name='num_input')

    cat_inputs, cat_embeds = [], []
    for i, (col, vocab_size) in enumerate(cat_vocab_sizes.items()):
        inp = layers.Input(shape=(1,), name=f'cat_{i}', dtype='int32')
        emb = layers.Embedding(vocab_size, min(50, (vocab_size + 1) // 2),
                               name='emb_' + col.replace(' ', '_'))(inp)
        emb = layers.Flatten()(emb)
        cat_inputs.append(inp)
        cat_embeds.append(emb)

    cat_concat = layers.Concatenate()(cat_embeds) if len(cat_embeds) > 1 else cat_embeds[0]
    cat_tiled  = layers.RepeatVector(lookback)(cat_concat)

    x = layers.Concatenate(axis=-1)([num_input, cat_tiled])
    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(horizon)(x)

    model = models.Model(inputs=[num_input] + cat_inputs, outputs=out)
    model.compile(optimizer='adam', loss='mse')
    return model

print('Model builder ready.')

## 3. Dataset & Evaluation Helpers

In [ ]:
def make_sequences(num_arr, cat_row, lookback, horizon, stride=7):
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, TARGET_IDX])
    X_num = np.array(X_num, dtype=np.float32)
    y     = np.array(y, dtype=np.float32)
    n     = len(X_num)
    X_cats = [np.full(n, cat_row[j], dtype=np.int32) for j in range(len(cat_row))]
    return X_num, X_cats, y


def build_global_arrays(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    y_tr, y_vl = [], []
    X_cats_tr = [[] for _ in CAT_COLS]
    X_cats_vl = [[] for _ in CAT_COLS]
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        cat_row   = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
        train_num = sdf[:TRAIN_END][NUM_COLS].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        Xn_tr, Xc_tr, yt = make_sequences(scaler.transform(train_num), cat_row, lookback, horizon)
        Xn_vl, Xc_vl, yv = make_sequences(scaler.transform(val_num),   cat_row, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)
        for j in range(len(CAT_COLS)):
            X_cats_tr[j].append(Xc_tr[j])
            X_cats_vl[j].append(Xc_vl[j])

    return (
        np.concatenate(X_num_tr), [np.concatenate(x) for x in X_cats_tr], np.concatenate(y_tr),
        np.concatenate(X_num_vl), [np.concatenate(x) for x in X_cats_vl], np.concatenate(y_vl),
        scalers
    )


def rolling_eval(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')

    cat_row     = sdf[ENC_COLS].iloc[0].values.astype(np.int32)
    full_scaled = scaler.transform(sdf[NUM_COLS].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num  = full_scaled[win_start:t_loc][np.newaxis]
        X_cats = [np.array([[cat_row[j]]], dtype=np.int32) for j in range(len(CAT_COLS))]

        fc_scaled = model.predict([X_num] + X_cats, verbose=0)[0]
        dummy = np.zeros((horizon, len(NUM_COLS)), dtype=np.float32)
        dummy[:, TARGET_IDX] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }


def run_ablation(ablation_name, model_name, lookback):
    """Train + rolling eval cho một lookback variant, trả về (summary_rows, detail_rows)."""
    os.makedirs(RESULT_DIR, exist_ok=True)
    summary_rows, detail_rows = [], []

    for h in HORIZONS:
        print(f'\n=== {ablation_name} | Horizon = {h} | Lookback = {lookback} ===')
        X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = build_global_arrays(h, lookback)
        print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')

        model = build_model(lookback, len(NUM_COLS), cat_vocab_sizes, horizon=h)
        cb = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model.fit(
            [X_num_tr] + X_cats_tr, y_tr,
            validation_data=([X_num_vl] + X_cats_vl, y_vl),
            epochs=50, batch_size=256,
            callbacks=[cb], verbose=0
        )

        print('  Rolling eval on TEST...')
        scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}
        for store, product in series_keys:
            key = f'{store}_{product}'
            r   = rolling_eval(model, scalers[key], store, product, h, lookback)
            for k in scores: scores[k].append(r[k])
            print(f"    {store} | {product} | sMAPE={r['smape']:.2f}% MASE={r['mase']:.4f} RMSE={r['rmse']:.2f} RMSLE={r['rmsle']:.4f}")
            detail_rows.append({
                'ablation': ablation_name, 'model': model_name,
                'store': store, 'product': product,
                'horizon': h, 'lookback': lookback,
                'smape': round(float(r['smape']), 4),
                'mase':  round(float(r['mase']),  4),
                'rmse':  round(float(r['rmse']),  4),
                'rmsle': round(float(r['rmsle']), 4),
            })

        row = {
            'ablation': ablation_name, 'model': model_name,
            'dataset': 'retail_inventory_daily', 'target': TARGET,
            'horizon': h, 'lookback': lookback,
            'mean_smape':   round(float(np.nanmean(scores['smape'])), 4),
            'median_smape': round(float(np.nanmedian(scores['smape'])), 4),
            'mean_mase':    round(float(np.nanmean(scores['mase'])), 4),
            'median_mase':  round(float(np.nanmedian(scores['mase'])), 4),
            'mean_rmse':    round(float(np.nanmean(scores['rmse'])), 4),
            'median_rmse':  round(float(np.nanmedian(scores['rmse'])), 4),
            'mean_rmsle':   round(float(np.nanmean(scores['rmsle'])), 4),
            'median_rmsle': round(float(np.nanmedian(scores['rmsle'])), 4),
        }
        summary_rows.append(row)
        print(f"  H={h} | sMAPE={row['mean_smape']:.2f}% MASE={row['mean_mase']:.4f} RMSE={row['mean_rmse']:.2f} RMSLE={row['mean_rmsle']:.4f}")

    return summary_rows, detail_rows

print('Helpers ready.')

---
# A8 — Lookback = 7

In [ ]:
summary_a8, detail_a8 = run_ablation('A8-Lookback7', 'LSTM-Lookback7', LOOKBACK_A8)

pd.DataFrame(summary_a8).to_csv(f'{RESULT_DIR}/ablation_A8_lookback7_summary.csv', index=False)
pd.DataFrame(detail_a8).to_csv(f'{RESULT_DIR}/ablation_A8_lookback7_details.csv',  index=False)
print(f'Saved A8 → {RESULT_DIR}')
pd.DataFrame(summary_a8)

---
# A9 — Lookback = 60

In [ ]:
summary_a9, detail_a9 = run_ablation('A9-Lookback60', 'LSTM-Lookback60', LOOKBACK_A9)

pd.DataFrame(summary_a9).to_csv(f'{RESULT_DIR}/ablation_A9_lookback60_summary.csv', index=False)
pd.DataFrame(detail_a9).to_csv(f'{RESULT_DIR}/ablation_A9_lookback60_details.csv',  index=False)
print(f'Saved A9 → {RESULT_DIR}')
pd.DataFrame(summary_a9)

---
# So sánh A8 / Proposed / A9

In [ ]:
import glob

# Load proposed result (GAO)
proposed_files = glob.glob(f'{RESULT_DIR}/*gao*summary*') + glob.glob(f'{RESULT_DIR}/*entity_emb_gao*')
if proposed_files:
    df_proposed = pd.read_csv(proposed_files[0])
    df_proposed['ablation'] = 'Proposed-Lookback30'
else:
    print('Proposed result not found — using placeholder.')
    df_proposed = None

compare_cols = ['ablation', 'horizon', 'lookback', 'mean_smape', 'mean_mase', 'mean_rmse', 'mean_rmsle']

frames = [
    pd.DataFrame(summary_a8)[compare_cols],
    pd.DataFrame(summary_a9)[compare_cols],
]
if df_proposed is not None:
    frames.insert(1, df_proposed[compare_cols])

df_compare = pd.concat(frames, ignore_index=True).sort_values(['horizon', 'lookback'])
print('\n=== Lookback Ablation: A8 (7) / Proposed (30) / A9 (60) ===')
print(df_compare.to_string(index=False))
print('\n→ Lookback tối ưu = lookback cho sMAPE thấp nhất nhất quán qua các horizon')